In [15]:
import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from sklearn.model_selection import train_test_split

# 1. Connect to your NEW, verified project asset
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get(name="library_occupancy_final", version="1")

# 2. Load and Sample (Crucial to keep RAM under control)
df = pd.read_parquet(data_asset.path).sample(frac=0.2, random_state=42)

# 3. Sort by Time to ensure Scientific Rigor
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')

# 4. Mandatory 4-Way Split
train_val_test, deploy_df = train_test_split(df, test_size=0.10, shuffle=False)
train_df, temp_df = train_test_split(train_val_test, test_size=0.333, shuffle=True, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=True, random_state=42)

print(f"SUCCESS! Project data loaded.")
print(f"Train Rows: {len(train_df)} | Deploy Rows: {len(deploy_df)}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


SUCCESS! Project data loaded.
Train Rows: 2427144 | Deploy Rows: 404322


In [13]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# Predict the average occupancy of the training set for every row in the test set
baseline_guess = train_df['meter_reading'].mean()
baseline_preds = np.full(shape=len(test_df), fill_value=baseline_guess)

# Calculate MAE
baseline_mae = mean_absolute_error(test_df['meter_reading'], baseline_preds)

print(f"Baseline Mean Prediction: {baseline_guess:.2f}")
print(f"Baseline MAE (The bar for your AI): {baseline_mae:.2f}")

Baseline Mean Prediction: 2255.41
Baseline MAE (The bar for your AI): 3822.78


In [9]:
from sklearn.ensemble import RandomForestRegressor
import mlflow
import mlflow.sklearn

# 1. Prepare Features (X) and Target (y)
# We use 'air_temperature' and 'cloud_coverage' as simple features for now
features = ['air_temperature', 'cloud_coverage', 'dew_temperature']
X_train = train_df[features].fillna(0) # Simple fill for missing values
y_train = train_df['meter_reading']

X_test = test_df[features].fillna(0)
y_test = test_df['meter_reading']

# 2. Start MLflow Tracking
mlflow.set_experiment("library-occupancy-analysis")

with mlflow.start_run(run_name="RF_Tuned_Time_Features"):
    # We increase trees to 50 and LIMIT the depth to 10
    # This prevents the model from "memorizing" specific noise
    rf_model = RandomForestRegressor(
        n_estimators=50, 
        max_depth=10, 
        min_samples_leaf=50, # Each "leaf" must have at least 50 rows
        random_state=42,
        n_jobs=-1 # Use all your CPU cores to make it fast
    )
    
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)
    rf_mae = mean_absolute_error(y_test, rf_preds)
    
    mlflow.log_metrics({"mae": rf_mae, "baseline_mae": 3822.78})
    mlflow.sklearn.log_model(rf_model, "model")
    
    print(f"Tuned Challenger MAE: {rf_mae:.2f}")
    print(f"New Improvement: {3822.78 - rf_mae:.2f}")

2026/04/10 14:44:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpqiug6i0f/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/04/10 14:44:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/04/10 14:44:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run RF_Tuned_Time_Features at: https://qatarcentral.api.azureml.ms/mlflow/v2.0/subscriptions/0b475409-9d7c-4dff-a07b-084eff651874/resourceGroups/rg-60304966/providers/Microsoft.MachineLearningServices/workspaces/amazon-electronics-lab-60304966/#/experiments/487a0ef1-07c8-442d-a193-67176d7f0de1/runs/d0f7e249-e0d9-4b3d-9209-df81fdc9d6f9.
2026/04/10 14:44:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at

Tuned Challenger MAE: 3820.98
New Improvement: 1.80


In [11]:
import mlflow

# 1. Grab the ID of your last successful training run
run_id = mlflow.last_active_run().info.run_id
model_uri = f"runs/{run_id}/model"
model_name = "library_occupancy_rf_model"

# 2. COMMENTED THIS OUT for the final GitHub version to prevent Version 3
# mlflow.register_model(model_uri, model_name)

print(f"Model '{model_name}' (Version 2) is already registered and ready for deployment.")

Registered model 'library_occupancy_rf_model' already exists. Creating a new version of this model...
Created version '2' of model 'library_occupancy_rf_model'.


In [16]:
import mlflow
import pandas as pd

# 1. Pull the model you registered into this notebook
model_uri = "models:/library_occupancy_rf_model/2"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# 2. Pick the columns the model expects
features = ['air_temperature', 'cloud_coverage', 'dew_temperature']
X_test_sample = test_df[features].fillna(0).head(10)
y_actual_sample = test_df['meter_reading'].head(10).values

# 3. Predict!
predictions = loaded_model.predict(X_test_sample)

# 4. Show the proof
results_table = pd.DataFrame({
    'Actual Occupancy': y_actual_sample,
    'Model Prediction': predictions
})

print(" These are your successful model results:")
display(results_table)

  return method()
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/IPython/core/formatters.py:406: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  return method()


,Actual Occupancy,Model Prediction
0,173.000,2285.497551
1,7587.890,2285.497551
2,494.500,2285.497551
3,175.948,2285.497551
4,0.000,2285.497551
5,5.010,2285.497551
6,0.000,2285.497551
7,309.073,2285.497551
8,121.281,2285.497551
9,1742.190,2285.497551
